# QC: IRF по месяцам — озеро vs Excel

Не запускает помесячный цикл `final_script_2`.

**Озеро:** `ods_alpha.scd1_trx` (SA / S01) + `ods_alpha.scd1_trx_int.n_amt_fee`.
Дата — `scd1_trx.d_trx_orig` (`trx_int` даты не имеет).
Это тот же периметр, из которого в `final_df` собирается `int_component`.

**Excel:** колонка «Комиссия МПС (IRF, ₽)» в месячных файлах `DATA_DIR`.
Августа в референсах обычно нет — будет только озеро.

Смотрите `int_coverage_pct` и `|IRF|` к предыдущему месяцу: провал покрытия при живом `trx_cnt` = дыра в `scd1_trx_int`, не «бизнес упал».

Костыль `run_august_irf_impute` подставляет **июль → август**. Если в озере дырявый июль, костыль **не** включать.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
period_start = '2026-01-01'
period_end_exclusive = '2026-09-01'

excel_header = 0
excel_header_by_month = {
    '2026-01': 1,
    '2026-02': 1,
    '2026-07': -1,
}
excel_reference_by_month = {
    '2026-01': DATA_DIR / '01_Январь_2026.xlsx',
    '2026-02': DATA_DIR / '02_Февраль_2026.xlsx',
    '2026-03': DATA_DIR / '03_Март_2026.xlsx',
    '2026-04': DATA_DIR / '04_Апрель_2026.xlsx',
    '2026-05': DATA_DIR / '05_Май_2026.xlsx',
    '2026-06': DATA_DIR / '06_Июнь_2026.xlsx',
    '2026-07': DATA_DIR / '07_Июль_2026.xlsx',
}

IRF_COL_NEEDLES = [
    'Комиссия МПС (IRF, ₽)',
    'Комиссия МПС (IRF, р)',
    'Комиссия МПС (IRF, руб)',
    'Комиссия МПС (IRF)',
    'IRF',
]
TRX_CNT_NEEDLES = ['Количество операций', 'Количеств операций', 'trx_cnt']
TRX_SUM_NEEDLES = ['Сумма операций', 'Сумма опреаций', 'trx_sum']

run_invalidate = True
output_xlsx = DATA_DIR / 'qc_irf_lake_vs_excel_2026_01_2026_08.xlsx'

print('DATA_DIR', DATA_DIR, 'exists=', DATA_DIR.exists())
for ym, p in excel_reference_by_month.items():
    print(f'  {ym}: exists={p.exists()}  {p.name}')

In [ ]:
def pick_col(columns, needles):
    ranked = []
    for col in columns:
        low = str(col).lower().replace('\n', ' ').strip()
        compact = re.sub(r'[^a-zа-я0-9]+', '', low)
        best = None
        for i, n in enumerate(needles):
            nlow = n.lower().strip()
            ncompact = re.sub(r'[^a-zа-я0-9]+', '', nlow)
            if low == nlow or compact == ncompact:
                score = (0, i)
            elif nlow in low or (ncompact and ncompact in compact):
                score = (1, i)
            else:
                continue
            if best is None or score < best:
                best = score
        if best is not None:
            ranked.append((best, col))
    if not ranked:
        return None
    ranked.sort(key=lambda x: x[0])
    return ranked[0][1]


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce',
    )


def resolve_excel_header(path, header):
    if header != -1:
        return header
    raw = pd.read_excel(path, header=None, nrows=12)
    markers = ('аур', 'амортизац', 'фин. рез', 'фин.рез', 'irf', 'комиссия мпс')
    for i, row in raw.iterrows():
        cells = ' '.join(str(x).lower().replace('\n', ' ') for x in row.tolist() if pd.notna(x))
        if any(m in cells for m in markers):
            print(f'  header=-1 → {i} ({path.name})')
            return i
    print(f'  header=-1 не найден, пробуем 0 ({path.name})')
    return 0


def load_excel_irf(path, header):
    hdr = resolve_excel_header(path, header)
    ex = pd.read_excel(path, header=hdr)
    irf_col = pick_col(ex.columns, IRF_COL_NEEDLES)
    if irf_col is None:
        raise ValueError(f'Нет колонки IRF в {path.name}. Колонки: {list(ex.columns)}')
    trx_cnt_col = pick_col(ex.columns, TRX_CNT_NEEDLES)
    trx_sum_col = pick_col(ex.columns, TRX_SUM_NEEDLES)
    irf = to_num_series(ex[irf_col])
    return {
        'irf_col': irf_col,
        'irf_excel': float(irf.fillna(0).sum()),
        'irf_excel_abs': float(irf.abs().fillna(0).sum()),
        'trx_cnt_excel': float(to_num_series(ex[trx_cnt_col]).fillna(0).sum()) if trx_cnt_col else np.nan,
        'trx_sum_excel': float(to_num_series(ex[trx_sum_col]).fillna(0).sum()) if trx_sum_col else np.nan,
        'excel_rows': int(len(ex)),
    }

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'},
)
imp._init_connection()
print('Impala connected')

if run_invalidate:
    with imp:
        for t in ('ods_alpha.scd1_trx', 'ods_alpha.scd1_trx_int'):
            try:
                imp.execute(f'invalidate metadata {t}')
                imp.execute(f'refresh {t}')
                print('[invalidate ok]', t)
            except Exception as exc:
                print('[invalidate fail]', t, type(exc).__name__)

In [ ]:
irf_probe_sql = f'''
SELECT
  cast(trunc(to_date(to_timestamp(t.d_trx_orig, 'yyyy-MM-dd HH:mm:ss')), 'MM') AS string) AS trx_month,
  count(*) AS trx_cnt,
  count(ti.n_trx) AS with_int_row,
  sum(cast(ti.n_amt_fee AS double)) AS sum_n_amt_fee,
  sum(abs(cast(ti.n_amt_fee AS double))) AS sum_n_amt_fee_abs,
  sum(CASE WHEN ti.n_trx IS NULL THEN 1 ELSE 0 END) AS trx_without_int
FROM ods_alpha.scd1_trx t
LEFT JOIN ods_alpha.scd1_trx_int ti ON ti.n_trx = t.n_trx
WHERE t.c_trx_class = 'SA'
  AND t.c_trx_type = 'S01'
  AND coalesce(t.cf_trx_stat, '') <> 'R'
  AND t.c_nter IS NOT NULL
  AND t.d_trx_orig >= '{period_start}'
  AND t.d_trx_orig <  '{period_end_exclusive}'
GROUP BY 1
ORDER BY 1
'''

print('Lake IRF', period_start, '…', period_end_exclusive)
with imp:
    try:
        imp.execute('set MEM_LIMIT=8g')
    except Exception:
        pass
    lake = imp.fetch(irf_probe_sql)

if lake is None or lake.empty:
    raise RuntimeError('Пустой probe озера')

lake['trx_month'] = lake['trx_month'].astype(str).str[:7]
for c in ('trx_cnt', 'with_int_row', 'sum_n_amt_fee', 'sum_n_amt_fee_abs', 'trx_without_int'):
    lake[c] = pd.to_numeric(lake[c], errors='coerce')
lake['int_coverage_pct'] = np.where(
    lake['trx_cnt'] > 0,
    100.0 * lake['with_int_row'] / lake['trx_cnt'],
    np.nan,
)
print('=== Озеро ===')
display(lake)

In [ ]:
excel_rows = []
for ym, path in excel_reference_by_month.items():
    hdr = excel_header_by_month.get(ym, excel_header)
    rec = {'trx_month': ym, 'excel_path': path.name, 'excel_exists': path.exists()}
    if not path.exists():
        rec['excel_error'] = 'file missing'
        excel_rows.append(rec)
        continue
    try:
        rec.update(load_excel_irf(path, hdr))
        rec['excel_error'] = ''
        print(f'OK {ym}: IRF={rec["irf_excel"]:,.2f}  col={rec["irf_col"]}')
    except Exception as exc:
        rec['excel_error'] = f'{type(exc).__name__}: {exc}'
        print('FAIL', ym, rec['excel_error'])
    excel_rows.append(rec)

excel_df = pd.DataFrame(excel_rows)
print('=== Excel ===')
display(excel_df)

In [ ]:
cmp = lake.merge(
    excel_df.drop(columns=['excel_path'], errors='ignore'),
    on='trx_month',
    how='outer',
)
cmp = cmp.sort_values('trx_month').reset_index(drop=True)

cmp['irf_lake_minus_excel'] = cmp['sum_n_amt_fee'] - cmp['irf_excel']
cmp['irf_abs_lake_minus_excel'] = cmp['sum_n_amt_fee_abs'] - cmp['irf_excel_abs']
cmp['irf_vs_excel_pct'] = np.where(
    cmp['irf_excel'].abs() > 1,
    100.0 * cmp['irf_lake_minus_excel'] / cmp['irf_excel'].abs(),
    np.nan,
)
prev_fee = cmp['sum_n_amt_fee_abs'].shift(1)
prev_trx = cmp['trx_cnt'].shift(1)
cmp['lake_irf_abs_mom'] = np.where(prev_fee.abs() > 1, cmp['sum_n_amt_fee_abs'] / prev_fee, np.nan)
cmp['lake_trx_mom'] = np.where(prev_trx > 0, cmp['trx_cnt'] / prev_trx, np.nan)

def _flag(r):
    if pd.isna(r.get('trx_cnt')):
        return 'нет месяца в озере'
    cov = r.get('int_coverage_pct')
    if pd.notna(cov) and cov < 5:
        return 'дыра озера (почти нет trx_int)'
    if pd.notna(cov) and cov < 50:
        return 'озеро неполное'
    if pd.isna(r.get('irf_excel')):
        return 'нет Excel'
    if pd.notna(r.get('irf_vs_excel_pct')) and abs(r['irf_vs_excel_pct']) > 30:
        return 'большое расхождение с Excel'
    return 'ок'

cmp['flag'] = cmp.apply(_flag, axis=1)

show = [
    'trx_month', 'trx_cnt', 'with_int_row', 'int_coverage_pct', 'trx_without_int',
    'sum_n_amt_fee', 'sum_n_amt_fee_abs', 'lake_trx_mom', 'lake_irf_abs_mom',
    'irf_excel', 'irf_excel_abs', 'irf_lake_minus_excel', 'irf_vs_excel_pct',
    'trx_cnt_excel', 'flag', 'excel_error',
]
show = [c for c in show if c in cmp.columns]
print('=== Сводка IRF: озеро vs Excel ===')
display(cmp[show])

print('\nКак читать:')
print('- trx живой, coverage < 5% → в scd1_trx_int нет месяца, не падение бизнеса')
print('- костыль июль→август имеет смысл, только если июль coverage высокий, август дырявый')
print('- знак: в Excel IRF часто «−», в озере как в источнике; смотрите и raw, и abs')

try:
    with pd.ExcelWriter(output_xlsx, engine='openpyxl') as w:
        lake.to_excel(w, sheet_name='lake', index=False)
        excel_df.to_excel(w, sheet_name='excel', index=False)
        cmp.to_excel(w, sheet_name='compare', index=False)
    print('Saved', output_xlsx)
except Exception as exc:
    print('Save xlsx failed:', type(exc).__name__, exc)